# 01 · Data Collection & Cleaning

This notebook **documents** the pipeline. It does not download anything: the work is done by scripts so it can be tested and rerun.

```
python -m src.data.fetch_pokemon     # Stage 1: PokéAPI -> data/raw/ (cached JSON)
python -m src.data.build_dataset     # Stages 2-3: parse + validate -> data/processed/pokemon.csv
```

**Source:** PokéAPI v2 (https://pokeapi.co), `pokemon-species` and `pokemon` endpoints. Fetched November 2026.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config import PROCESSED_DIR, RAW_DIR
from src.data.build_dataset import validate
from src.data.form_overrides import FORM_OVERRIDES

df = pd.read_csv(PROCESSED_DIR / "pokemon.csv")
print("Rows:", len(df), "| Columns:", df.shape[1])
print("Raw cache present:", RAW_DIR.exists())
df.dtypes.to_frame("type").T

Rows: 1025 | Columns: 24
Raw cache present: True


,dex_id,name,species_slug,pokemon_slug,generation,type_1,type_2,hp,attack,defense,...,abilities,hidden_ability,evolves_from,is_legendary,is_mythical,sprite_url,artwork_url,form_override,evolution_stage,is_fully_evolved
type,int64,str,str,str,int64,str,str,int64,int64,int64,...,str,str,str,bool,bool,str,str,bool,int64,bool


## Cleaning decisions

| Decision | What | Why |
|---|---|---|
| One row per species | Default form from PokéAPI (`is_default`) | Alternate forms (Megas, regional forms, Gigantamax) are excluded in this version |
| Battle-form overrides | See table below | Rule: default form is a temporary state left automatically on entering battle, no player action |
| Scope | Generations I–IX | `MAX_GENERATION = 9`; nothing excluded today (1,025 species) |
| Missing stats | None found | Validation would stop the pipeline instead of dropping rows |
| Duplicates | None found | Dex ID and name both checked unique |
| Evolution stage | Derived from `evolves_from` links | No extra API calls; baby Pokémon count as stage 1 |

In [2]:
pd.DataFrame(
    [(s, f, df.loc[df["species_slug"] == s, "bst"].item()) for s, f in FORM_OVERRIDES.items()],
    columns=["species", "form used", "bst of form used"],
)

,species,form used,bst of form used
0,wishiwashi,wishiwashi-school,620
1,terapagos,terapagos-terastal,600


In [3]:
validate(df)   # same checks the pipeline runs; raises an error if anything is wrong
print(df["is_fully_evolved"].value_counts().rename({True: "fully evolved", False: "not fully evolved"}))
print(df["evolution_stage"].value_counts().sort_index().rename("species per stage"))

Validation passed
is_fully_evolved
fully evolved        568
not fully evolved    457
Name: count, dtype: int64
evolution_stage
1    541
2    363
3    121
Name: species per stage, dtype: int64


## Known limitations

- Base stats are the **current** values; some Pokémon had different stats in older games.
- Types are current (Fairy type exists since Gen VI and is applied to older Pokémon).
- Default forms do not always represent competitive use (e.g. Zacian is used in Crowned form, which needs a held item). See `src/data/form_overrides.py`.